In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys

print("at directory:", os.getcwd())
print("changing to root directory")
os.chdir("..")
print("at directory:", os.getcwd())

at directory: /Users/wilka/git/research/human-dyna-web/analysis_notebooks
changing to root directory
at directory: /Users/wilka/git/research/human-dyna-web


In [3]:
import glob

print("at directory:", os.getcwd())

glob.glob("keys/*")

at directory: /Users/wilka/git/research/human-dyna-web


['keys/datastore-key.json']

In [4]:
# ON COMPUTER
RESULTS_DIR = "/Users/wilka/git/research/"
USER_RESULTS_DIR = os.path.join(RESULTS_DIR, "results/human_dyna/user_data/exps")

In [5]:
# from analysis import housemaze_download_data
from analysis.download_user_data import download_user_files

bucket_name = "human-dyna"
bucket_prefix = "data/"
# pattern = "data/data_user=*_name=r0-v2*debug=0.json"
# human_data_pattern = "v2*shortcut*debug-2"
# human_data_pattern = "pilot*plan*"
human_data_pattern = "final*v2*"

download_user_files(
  bucket_name=bucket_name,
  prefix=bucket_prefix,
  pattern=f"data/*user=*{human_data_pattern}*",
  destination_folder=USER_RESULTS_DIR,
)

File already exists: 
	 /Users/wilka/git/research/results/human_dyna/user_data/exps/user=1001779639_name=jaxmaze_final-v2-r1-t0-paths_debug=0.json
File already exists: 
	 /Users/wilka/git/research/results/human_dyna/user_data/exps/user=1008223757_name=jaxmaze_final-v2-r0-t0-plan_debug=0.json
File already exists: 
	 /Users/wilka/git/research/results/human_dyna/user_data/exps/user=1017609201_name=jaxmaze_final-v2-r1-t0-start_debug=0.json
File already exists: 
	 /Users/wilka/git/research/results/human_dyna/user_data/exps/user=1047131165_name=jaxmaze_final-v2-r1-t0-paths_debug=0.json
File already exists: 
	 /Users/wilka/git/research/results/human_dyna/user_data/exps/user=1052354210_name=jaxmaze_final-v2-r1-t0-shortcut_debug=0.json
File already exists: 
	 /Users/wilka/git/research/results/human_dyna/user_data/exps/user=1064468626_name=jaxmaze_final-v2-r1-t0-paths_debug=0.json
File already exists: 
	 /Users/wilka/git/research/results/human_dyna/user_data/exps/user=1064862666_name=jaxmaze_fin

In [6]:
from glob import glob

files = f"{USER_RESULTS_DIR}/*{human_data_pattern}*.json"
files = f"{USER_RESULTS_DIR}/*path*.json"
# files = 'jaxemaze_data/*.json'
valid_files = list(set(glob(files)))
len(valid_files)

139

In [7]:
from pprint import pprint
import json
from nicewebrl.utils import read_all_records_sync

nfinished = 0
for idx, file in enumerate(valid_files):
  try:
    data = read_all_records_sync(file)
  except Exception as e:
    print("-" * 25)
    print(idx, os.path.basename(file))
    print(e)
    continue
  if len(data) < 2:
    print(f"{file} has length = {len(data)}")
    continue
  finished = data[-1].get("finished", False)
  if finished:
    print
    nfinished += 1
    print("-" * 25)
    print(idx, os.path.basename(file))
    print("\n" + data[-1]["feedback"] or "N/A")
    # print('bonus:', data[-1]['bonus'])
    # print("feedback:")
    if "feedback" in data[-2]["data"]:
      pprint(data[-2]["data"])
  break

Incomplete record in /Users/wilka/git/research/results/human_dyna/user_data/exps/user=1113634923_name=exp5-v2-r0-t0-paths_debug=0.json


/Users/wilka/git/research/results/human_dyna/user_data/exps/user=1113634923_name=exp5-v2-r0-t0-paths_debug=0.json has length = 0
-------------------------
1 user=3109174614_name=jaxmaze_final-v2-r1-t0-paths_debug=0.json

The experiement went fine


In [9]:
# data[-1]

In [8]:
# import nicewebrl

# # file = '/Users/wilka/git/research/results/human_dyna/user_data/exp4/data_user=1844620330_name=exp4-v1-r1-t0-plan_exp=4_debug=0.json'
# # file = "/Users/wilka/git/research/results/human_dyna//user_data/exp4/data_user=708665372_name=exp4-v1-r0-t0-plan_exp=4_debug=0.json"

# data = nicewebrl.read_all_records(file, synchronous=True)

In [19]:
# data[0]["data"].keys(), data[-2]["data"].keys()

In [10]:
import jax
from experiment_utils import SuccessTrackingAutoResetWrapper
from analysis.housemaze_user_data import (
  make_env_params,
  task_objects,
  make_episode_data,
)
from housemaze.human_dyna import multitask_env
from housemaze.human_dyna import web_env
from housemaze.human_dyna import mazes

################
# Setup environment
################
dummy_rng = jax.random.PRNGKey(42)
dummy_env_params = make_env_params(mazes.big_practice_maze)
task_runner = multitask_env.TaskRunner(task_objects=task_objects)
base_env = web_env.HouseMaze(
  task_runner=task_runner,
  num_categories=200,
)
end = SuccessTrackingAutoResetWrapper(base_env)
example_web_timestep = end.reset(dummy_rng, dummy_env_params)

No file specified for image dict.
Using: /Users/wilka/git/research/human-dyna-web/libraries/housemaze/housemaze/image_data.pkl


In [13]:
# TESTING
df, data = make_episode_data(
  file=file,
  example_timestep=example_web_timestep,
  overwrite_episode_info=True,
  require_finished=False,
)
df.head()

No file specified for image dict.
Using: /Users/wilka/git/research/human-dyna-web/libraries/housemaze/housemaze/image_data.pkl


index,maze,condition,name,block,manipulation,global_episode_idx,episode_idx,eval,task,room,user_id,age,sex,worker_id,hit_id,assignment_id,git_version,user,debug,exp_name,tell_reuse,reversal,optimal_length,success,path_length,termination,log_first_rt,first_rt,log_avg_rt,log_total_rt,total_rt,log_avg_post_rt,log_max_rt,log_max_post_rt,log_max_init_post_rt,log_max_end_rt,log_max_final_rt,optimal_length_deviance,reuse
u32,str,i64,str,str,i64,i64,i64,bool,i64,i64,i64,i64,str,str,str,str,str,str,str,str,i64,str,i64,f32,i64,f32,f32,f64,f32,f32,f64,f32,f32,f32,f32,f32,f32,i64,i32
0,"""big_m3_maze1_(F,F)""",0,"""big_m3_maze1__(F,F)_eval1""","""reusing longer of two paths wh…",3,0,1,false,31,0,3109174614,39,"""Female""","""A4TE7LF9CEVGA""","""3W1K7D6QSB37EFSH9N9XQPSS7CUBZL""","""3CCZ6YKWR85S5SRPVHTNZYM1WER95J""","""git_version_unknown""","""3109174614""","""0""","""jaxmaze""",1,"""F,F""",9,1.0,10,1.0,0.412116,1.51,-0.706478,-7.064776,6.11,-0.830766,0.412116,0.13803,-0.009031,0.412116,0.13803,1,null
1,"""big_m3_maze1_(F,F)""",0,"""big_m3_maze1__(F,F)_eval1""","""reusing longer of two paths wh…",3,1,2,false,31,0,3109174614,39,"""Female""","""A4TE7LF9CEVGA""","""3W1K7D6QSB37EFSH9N9XQPSS7CUBZL""","""3CCZ6YKWR85S5SRPVHTNZYM1WER95J""","""git_version_unknown""","""3109174614""","""0""","""jaxmaze""",1,"""F,F""",26,1.0,35,1.0,-0.534418,0.586,-1.31606,-46.062107,10.492001,-1.33905,-0.018154,-0.018154,-0.018154,-0.921278,-0.921278,9,null
2,"""big_m3_maze1_(F,F)""",0,"""big_m3_maze1__(F,F)_eval1""","""reusing longer of two paths wh…",3,2,3,false,46,1,3109174614,39,"""Female""","""A4TE7LF9CEVGA""","""3W1K7D6QSB37EFSH9N9XQPSS7CUBZL""","""3CCZ6YKWR85S5SRPVHTNZYM1WER95J""","""git_version_unknown""","""3109174614""","""0""","""jaxmaze""",1,"""F,F""",24,1.0,27,1.0,-0.45097,0.637,-1.266097,-34.184624,11.862,-1.297448,1.299377,1.299377,1.299377,-0.857998,-0.857998,3,null
3,"""big_m3_maze1_(F,F)""",0,"""big_m3_maze1__(F,F)_eval1""","""reusing longer of two paths wh…",3,3,4,false,46,1,3109174614,39,"""Female""","""A4TE7LF9CEVGA""","""3W1K7D6QSB37EFSH9N9XQPSS7CUBZL""","""3CCZ6YKWR85S5SRPVHTNZYM1WER95J""","""git_version_unknown""","""3109174614""","""0""","""jaxmaze""",1,"""F,F""",45,1.0,54,1.0,-0.825514,0.438,-1.580533,-85.348785,11.817,-1.594779,-0.825514,-0.860359,-0.860359,-1.207278,-1.002366,9,null
4,"""big_m3_maze1_(F,F)""",0,"""big_m3_maze1__(F,F)_eval1""","""reusing longer of two paths wh…",3,4,5,false,46,1,3109174614,39,"""Female""","""A4TE7LF9CEVGA""","""3W1K7D6QSB37EFSH9N9XQPSS7CUBZL""","""3CCZ6YKWR85S5SRPVHTNZYM1WER95J""","""git_version_unknown""","""3109174614""","""0""","""jaxmaze""",1,"""F,F""",24,1.0,28,1.0,-0.365269,0.694,-1.293849,-36.22776,8.167999,-1.32824,-0.365269,-0.778683,-0.778683,-1.016083,-0.857998,4,null


In [22]:
df['reuse'].unique()

reuse
i32
null
0
1
